# Notebook 01 - Eksplorasi & Preprocessing

**Tujuan:** memahami struktur data CIC-IDS2017, mendefinisikan partisi Non-IID per node,
dan menyiapkan data bersih + terstandarisasi untuk training ANN serta federated learning.

**Ringkas alur:** daftar file CSV, cek kolom & label, definisi Non-IID, bersihkan NaN/Inf,
drop zero variance, encoding label, split train/test, standardisasi, simpan data preprocessed.

**Catatan evaluasi:** split 80/20 stratified tanpa validation terpisah (bisa ditambah jika diperlukan).


In [1]:
import pandas as pd
import numpy as np
import os

# path ke folder dataset CIC-IDS2017
DATA_DIR = "../datasets/cic-ids2017"

In [2]:
RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
print("Random seed set:", RANDOM_SEED)

Random seed set: 42


In [3]:
# ambil semua file CSV yang tersedia di folder dataset
files = sorted([f for f in os.listdir(DATA_DIR) if f.endswith(".csv")])
for f in files:
    print(f)

Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv
Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv
Friday-WorkingHours-Morning.pcap_ISCX.csv
Monday-WorkingHours.pcap_ISCX.csv
Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv
Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv
Tuesday-WorkingHours.pcap_ISCX.csv
Wednesday-workingHours.pcap_ISCX.csv


In [4]:
# load satu file sebagai sampel untuk inspeksi awal struktur kolom
df_sample = pd.read_csv(
    os.path.join(DATA_DIR, "Wednesday-workingHours.pcap_ISCX.csv"),
    low_memory=False
)
# bersihkan leading/trailing spasi dari nama kolom
# ini quirk terkenal di CIC-IDS2017
df_sample.columns = df_sample.columns.str.strip()

print("Shape:", df_sample.shape)
print("\nKolom:")
print(df_sample.columns.tolist())

Shape: (692703, 79)

Kolom:
['Destination Port', 'Flow Duration', 'Total Fwd Packets', 'Total Backward Packets', 'Total Length of Fwd Packets', 'Total Length of Bwd Packets', 'Fwd Packet Length Max', 'Fwd Packet Length Min', 'Fwd Packet Length Mean', 'Fwd Packet Length Std', 'Bwd Packet Length Max', 'Bwd Packet Length Min', 'Bwd Packet Length Mean', 'Bwd Packet Length Std', 'Flow Bytes/s', 'Flow Packets/s', 'Flow IAT Mean', 'Flow IAT Std', 'Flow IAT Max', 'Flow IAT Min', 'Fwd IAT Total', 'Fwd IAT Mean', 'Fwd IAT Std', 'Fwd IAT Max', 'Fwd IAT Min', 'Bwd IAT Total', 'Bwd IAT Mean', 'Bwd IAT Std', 'Bwd IAT Max', 'Bwd IAT Min', 'Fwd PSH Flags', 'Bwd PSH Flags', 'Fwd URG Flags', 'Bwd URG Flags', 'Fwd Header Length', 'Bwd Header Length', 'Fwd Packets/s', 'Bwd Packets/s', 'Min Packet Length', 'Max Packet Length', 'Packet Length Mean', 'Packet Length Std', 'Packet Length Variance', 'FIN Flag Count', 'SYN Flag Count', 'RST Flag Count', 'PSH Flag Count', 'ACK Flag Count', 'URG Flag Count', 'CWE 

In [5]:
# cek distribusi label di setiap file
# hanya load kolom Label untuk efisiensi memori
for fname in files:
    df_temp = pd.read_csv(
        os.path.join(DATA_DIR, fname),
        low_memory=False,
        usecols=lambda c: c.strip() == "Label"
    )
    df_temp.columns = df_temp.columns.str.strip()
    df_temp["Label"] = df_temp["Label"].str.strip()
    print(f"\n{fname}")
    print(df_temp["Label"].value_counts())


Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv
Label
DDoS      128027
BENIGN     97718
Name: count, dtype: int64

Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv
Label
PortScan    158930
BENIGN      127537
Name: count, dtype: int64

Friday-WorkingHours-Morning.pcap_ISCX.csv
Label
BENIGN    189067
Bot         1966
Name: count, dtype: int64

Monday-WorkingHours.pcap_ISCX.csv
Label
BENIGN    529918
Name: count, dtype: int64

Thursday-WorkingHours-Afternoon-Infilteration.pcap_ISCX.csv
Label
BENIGN          288566
Infiltration        36
Name: count, dtype: int64

Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv
Label
BENIGN                        168186
Web Attack � Brute Force        1507
Web Attack � XSS                 652
Web Attack � Sql Injection        21
Name: count, dtype: int64

Tuesday-WorkingHours.pcap_ISCX.csv
Label
BENIGN         432074
FTP-Patator      7938
SSH-Patator      5897
Name: count, dtype: int64

Wednesday-workingHours.pcap_ISCX.csv
Label
BENIGN        

In [6]:
# definisi partisi Non-IID: tiap node dapat file hari yang berbeda
# Non-IID dipilih karena lebih realistis, tiap node merepresentasikan
# segmen jaringan yang menghadapi ancaman berbeda
NODE_FILES = {
    "node1": ["Tuesday-WorkingHours.pcap_ISCX.csv"],
    "node2": ["Wednesday-workingHours.pcap_ISCX.csv"],
    "node3": [
        "Friday-WorkingHours-Afternoon-DDos.pcap_ISCX.csv",
        "Friday-WorkingHours-Morning.pcap_ISCX.csv",
        "Thursday-WorkingHours-Morning-WebAttacks.pcap_ISCX.csv",
        "Friday-WorkingHours-Afternoon-PortScan.pcap_ISCX.csv",
    ]
}

# kelas dengan sampel terlalu sedikit untuk dipelajari model
# Heartbleed: 11 sampel, Infiltration: 36 sampel
DROP_LABELS = {"Heartbleed", "Infiltration"}

# gabungkan semua varian Web Attack menjadi satu kelas
# karakter \ufffd muncul karena encoding Windows-1252 dibaca sebagai UTF-8
WEB_ATTACK_LABELS = {
    "Web Attack \ufffd Brute Force": "Web Attack",
    "Web Attack \ufffd XSS": "Web Attack",
    "Web Attack \ufffd Sql Injection": "Web Attack",
}

def load_node_data(file_list):
    dfs = []
    for fname in file_list:
        df = pd.read_csv(
            os.path.join(DATA_DIR, fname),
            low_memory=False
        )
        # bersihkan spasi dari nama kolom dan nilai label
        df.columns = df.columns.str.strip()
        df["Label"] = df["Label"].str.strip()
        dfs.append(df)

    df = pd.concat(dfs, ignore_index=True)

    # terapkan mapping web attack dan drop kelas minoritas ekstrem
    df["Label"] = df["Label"].replace(WEB_ATTACK_LABELS)
    df = df[~df["Label"].isin(DROP_LABELS)]

    return df

In [7]:
# load dan verifikasi distribusi label tiap node
df_node1 = load_node_data(NODE_FILES["node1"])
print("Node 1 shape:", df_node1.shape)
print(df_node1["Label"].value_counts())

df_node2 = load_node_data(NODE_FILES["node2"])
print("\nNode 2 shape:", df_node2.shape)
print(df_node2["Label"].value_counts())

df_node3 = load_node_data(NODE_FILES["node3"])
print("\nNode 3 shape:", df_node3.shape)
print(df_node3["Label"].value_counts())

Node 1 shape: (445909, 79)
Label
BENIGN         432074
FTP-Patator      7938
SSH-Patator      5897
Name: count, dtype: int64

Node 2 shape: (692692, 79)
Label
BENIGN              440031
DoS Hulk            231073
DoS GoldenEye        10293
DoS slowloris         5796
DoS Slowhttptest      5499
Name: count, dtype: int64

Node 3 shape: (873611, 79)
Label
BENIGN        582508
PortScan      158930
DDoS          128027
Web Attack      2180
Bot             1966
Name: count, dtype: int64


In [8]:
# cek jumlah NaN dan Inf di tiap node
# NaN dan Inf adalah masalah umum di CIC-IDS2017 akibat pembagian dengan nol
# saat kalkulasi flow rate
for name, df in [("Node 1", df_node1), ("Node 2", df_node2), ("Node 3", df_node3)]:
    nan_count = df.isnull().sum().sum()
    num_cols = df.select_dtypes(include=[np.number]).columns
    inf_count = np.isinf(df[num_cols]).sum().sum()
    print(f"{name}: NaN={nan_count}, Inf={inf_count}")

Node 1: NaN=201, Inf=327
Node 2: NaN=1008, Inf=1586
Node 3: NaN=67, Inf=1257


In [9]:
# bersihkan NaN dan Inf dengan drop baris
# strategi drop dipilih karena jumlahnya sangat kecil (< 0.4%)
# sehingga tidak mempengaruhi distribusi data secara signifikan
def clean_node_data(df):
    num_cols = df.select_dtypes(include=[np.number]).columns
    # ganti Inf dengan NaN supaya bisa di-drop sekaligus
    df[num_cols] = df[num_cols].replace([np.inf, -np.inf], np.nan)
    before = len(df)
    df = df.dropna()
    after = len(df)
    print(f"Dropped {before - after} baris ({(before-after)/before*100:.3f}%)")
    return df

df_node1 = clean_node_data(df_node1)
print("Node 1:", df_node1.shape)
print(df_node1["Label"].value_counts())

df_node2 = clean_node_data(df_node2)
print("\nNode 2:", df_node2.shape)
print(df_node2["Label"].value_counts())

df_node3 = clean_node_data(df_node3)
print("\nNode 3:", df_node3.shape)
print(df_node3["Label"].value_counts())

Dropped 264 baris (0.059%)
Node 1: (445645, 79)
Label
BENIGN         431813
FTP-Patator      7935
SSH-Patator      5897
Name: count, dtype: int64
Dropped 1297 baris (0.187%)

Node 2: (691395, 79)
Label
BENIGN              439683
DoS Hulk            230124
DoS GoldenEye        10293
DoS slowloris         5796
DoS Slowhttptest      5499
Name: count, dtype: int64
Dropped 662 baris (0.076%)

Node 3: (872949, 79)
Label
BENIGN        581984
PortScan      158804
DDoS          128025
Web Attack      2180
Bot             1956
Name: count, dtype: int64


In [10]:
# identifikasi kolom zero variance: nilai konstan di seluruh dataset
# kolom ini tidak memberi informasi apapun ke model dan harus di-drop
for name, df in [("Node 1", df_node1), ("Node 2", df_node2), ("Node 3", df_node3)]:
    num_cols = df.select_dtypes(include=[np.number]).columns
    zero_var = [col for col in num_cols if df[col].std() == 0]
    print(f"{name} - kolom zero variance: {zero_var}")

print("\nKolom duplikat:", df_node1.columns[df_node1.columns.duplicated()].tolist())

Node 1 - kolom zero variance: ['Bwd PSH Flags', 'Fwd URG Flags', 'Bwd URG Flags', 'CWE Flag Count', 'Fwd Avg Bytes/Bulk', 'Fwd Avg Packets/Bulk', 'Fwd Avg Bulk Rate', 'Bwd Avg Bytes/Bulk', 'Bwd Avg Packets/Bulk', 'Bwd Avg Bulk Rate']
Node 2 - kolom zero variance: ['Bwd PSH Flags', 'Fwd URG Flags', 'Bwd URG Flags', 'CWE Flag Count', 'Fwd Avg Bytes/Bulk', 'Fwd Avg Packets/Bulk', 'Fwd Avg Bulk Rate', 'Bwd Avg Bytes/Bulk', 'Bwd Avg Packets/Bulk', 'Bwd Avg Bulk Rate']
Node 3 - kolom zero variance: ['Bwd PSH Flags', 'Fwd URG Flags', 'Bwd URG Flags', 'CWE Flag Count', 'Fwd Avg Bytes/Bulk', 'Fwd Avg Packets/Bulk', 'Fwd Avg Bulk Rate', 'Bwd Avg Bytes/Bulk', 'Bwd Avg Packets/Bulk', 'Bwd Avg Bulk Rate']

Kolom duplikat: []


In [11]:
# drop 10 kolom zero variance yang ditemukan di semua node
ZERO_VAR_COLS = [
    'Bwd PSH Flags', 'Fwd URG Flags', 'Bwd URG Flags', 'CWE Flag Count',
    'Fwd Avg Bytes/Bulk', 'Fwd Avg Packets/Bulk', 'Fwd Avg Bulk Rate',
    'Bwd Avg Bytes/Bulk', 'Bwd Avg Packets/Bulk', 'Bwd Avg Bulk Rate'
]

def drop_zero_var(df):
    df = df.drop(columns=ZERO_VAR_COLS)
    return df

df_node1 = drop_zero_var(df_node1)
df_node2 = drop_zero_var(df_node2)
df_node3 = drop_zero_var(df_node3)

print("Kolom setelah drop:", df_node1.shape[1])
print("Fitur:", df_node1.shape[1] - 1, "(minus kolom Label)")

Kolom setelah drop: 69
Fitur: 68 (minus kolom Label)


In [12]:
from sklearn.preprocessing import LabelEncoder

# fit label encoder di gabungan semua label dari ketiga node
# penting: encoder harus di-fit di semua node sekaligus
# supaya mapping integer konsisten di semua node
all_labels = pd.concat([
    df_node1["Label"],
    df_node2["Label"],
    df_node3["Label"]
])

le = LabelEncoder()
le.fit(all_labels)

print("Mapping label:")
for i, cls in enumerate(le.classes_):
    print(f"  {i}: {cls}")

df_node1["Label"] = le.transform(df_node1["Label"])
df_node2["Label"] = le.transform(df_node2["Label"])
df_node3["Label"] = le.transform(df_node3["Label"])

Mapping label:
  0: BENIGN
  1: Bot
  2: DDoS
  3: DoS GoldenEye
  4: DoS Hulk
  5: DoS Slowhttptest
  6: DoS slowloris
  7: FTP-Patator
  8: PortScan
  9: SSH-Patator
  10: Web Attack


In [13]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

def split_and_scale(df):
    X = df.drop(columns=["Label"]).values
    y = df["Label"].values

    # split 80% train, 20% test dengan stratify supaya proporsi kelas terjaga
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42, stratify=y
    )

    # StandardScaler: normalisasi fitur ke mean=0, std=1
    # fit hanya di training data untuk menghindari data leakage
    # tiap node fit scaler sendiri sesuai prinsip FL
    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train)
    X_test = scaler.transform(X_test)

    return X_train, X_test, y_train, y_test, scaler

X_train1, X_test1, y_train1, y_test1, scaler1 = split_and_scale(df_node1)
X_train2, X_test2, y_train2, y_test2, scaler2 = split_and_scale(df_node2)
X_train3, X_test3, y_train3, y_test3, scaler3 = split_and_scale(df_node3)

print("Node 1 train:", X_train1.shape, "test:", X_test1.shape)
print("Node 2 train:", X_train2.shape, "test:", X_test2.shape)
print("Node 3 train:", X_train3.shape, "test:", X_test3.shape)

Node 1 train: (356516, 68) test: (89129, 68)
Node 2 train: (553116, 68) test: (138279, 68)
Node 3 train: (698359, 68) test: (174590, 68)


In [14]:
import pickle

SAVE_DIR = "../datasets/preprocessed"
os.makedirs(SAVE_DIR, exist_ok=True)

# simpan data preprocessed ke disk supaya tidak perlu preprocessing ulang
# di notebook berikutnya
for node_id, (X_train, X_test, y_train, y_test) in enumerate([
    (X_train1, X_test1, y_train1, y_test1),
    (X_train2, X_test2, y_train2, y_test2),
    (X_train3, X_test3, y_train3, y_test3),
], start=1):
    np.save(f"{SAVE_DIR}/node{node_id}_X_train.npy", X_train)
    np.save(f"{SAVE_DIR}/node{node_id}_X_test.npy", X_test)
    np.save(f"{SAVE_DIR}/node{node_id}_y_train.npy", y_train)
    np.save(f"{SAVE_DIR}/node{node_id}_y_test.npy", y_test)

# simpan label encoder untuk digunakan di notebook evaluasi
with open(f"{SAVE_DIR}/label_encoder.pkl", "wb") as f:
    pickle.dump(le, f)

print("Semua data tersimpan di:", SAVE_DIR)
print(os.listdir(SAVE_DIR))

Semua data tersimpan di: ../datasets/preprocessed
['node3_y_train.npy', 'node2_y_train.npy', 'node3_X_test.npy', 'node2_y_test.npy', 'label_encoder.pkl', 'node2_X_train.npy', 'node1_X_test.npy', 'node1_y_test.npy', 'node1_X_train.npy', 'node3_y_test.npy', 'node3_X_train.npy', 'node2_X_test.npy', 'node1_y_train.npy']
